# 2.1 · 描述统计 / Descriptive Statistics

> **课程定位 / Where this fits**
> **Part 2 第 1 课**。拿到任何数据，建模之前的第一件事是"描述它"——位置、散布、形状、关联。本课把这四类统计量的**公式、直觉、陷阱**一次讲透。
> **Part 2, lesson 1.** Before modeling anything: describe it — location, spread, shape, association.

> 📐 **符号约定**（见 [`NOTATION.md`](../NOTATION.md)）：$n$ 样本数，$\bar{x}$ 样本均值，$s$ 样本标准差，$Q_1/Q_2/Q_3$ 四分位数。

> 💡 **面试相关**
> - "均值 vs 中位数什么时候差很大" ★★★★（偏态）
> - "为什么样本方差除 $n-1$" ★★★★
> - "Pearson vs Spearman 相关" ★★★★
> - "Anscombe 四重奏说明什么" ★★★

---

## 目录
1. [位置：均值 / 中位数 / 众数 / 截尾均值](#1)
2. [散布：方差 / 标准差 / IQR / MAD](#2)
3. [⭐ 为什么除 n-1（无偏性证明 + 模拟）](#3)
4. [形状：偏度与峰度](#4)
5. [标准化与 z 分数](#5)
6. [关联：Pearson / Spearman / Kendall ⭐](#6)
7. [⚠ Anscombe 四重奏：只看数字会死](#7)
8. [实战：Tips 数据集完整描述报告](#8)
9. [小结](#9)


<a id="1"></a>
## 1. 位置 / Location

| 统计量 | 公式 / 定义 | 稳健性 / Robustness |
|---|---|---|
| 均值 / Mean | $\bar{x} = \frac{1}{n}\sum_i x_i$ | ❌ 一个异常值就被拖走 |
| 中位数 / Median | 排序后中间值 | ✅ 击穿点 50% |
| 众数 / Mode | 出现最多的值 | 适合类别变量 |
| 截尾均值 / Trimmed mean | 去掉两端各 $\alpha\%$ 再平均 | ⭐ 折中（奥运评分制）|

**击穿点 / breakdown point** = 要"污染"多大比例的数据才能让统计量任意坏。均值是 $0$（1 个点就够），中位数是 $50\%$。
Breakdown point = fraction of contamination needed to ruin the statistic. Mean: 0; median: 50%.


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# 演示稳健性：一个亿万富翁走进酒吧 / A billionaire walks into a bar
incomes = np.array([45, 52, 38, 61, 48, 55, 42, 58, 50, 47])    # 单位: k$
incomes_with_musk = np.append(incomes, 50_000)                   # +1 个异常值

for name, data in [("正常酒吧", incomes), ("富翁进来后", incomes_with_musk)]:
    print(f"{name:<8} mean={np.mean(data):>8.1f}k   median={np.median(data):>6.1f}k   "
          f"trimmed(10%)={st.trim_mean(data, 0.1):>6.1f}k")


**均值被拖到 4586k（4.5M）**，但中位数几乎没动——这就是为什么**收入、房价、用户消费这类右偏数据，报告里永远用中位数**。
The mean explodes; the median barely moves. This is why income/house-price/spend reports always use medians.


<a id="2"></a>
## 2. 散布 / Spread

| 统计量 | 公式 | 特点 |
|---|---|---|
| 样本方差 | $s^2 = \frac{1}{n-1}\sum_i (x_i - \bar{x})^2$ | 单位是平方，难解释 |
| 标准差 | $s = \sqrt{s^2}$ | 和数据同单位 ⭐ |
| 极差 / Range | $\max - \min$ | 最不稳健（只看两个点）|
| IQR | $Q_3 - Q_1$ | 稳健；箱线图的盒子 |
| MAD | $\text{median}(\lvert x_i - \text{median}(x)\rvert)$ | 最稳健的散布度量 |

**MAD → σ 换算**：正态数据下 $\hat{\sigma} = 1.4826 \times \text{MAD}$（稳健标准差估计，异常检测常用）。
For normal data, $\hat\sigma = 1.4826 \cdot \text{MAD}$ — the robust sigma used in outlier detection.


In [ ]:
data = rng.normal(loc=100, scale=15, size=1000)
data_dirty = np.append(data, [500, 600, -200])      # 加 3 个异常值 / add 3 outliers

for name, d in [("干净数据", data), ("带异常值", data_dirty)]:
    mad = np.median(np.abs(d - np.median(d)))
    print(f"{name}:  std={np.std(d, ddof=1):>7.2f}   IQR={np.percentile(d,75)-np.percentile(d,25):>6.2f}   "
          f"1.4826*MAD={1.4826*mad:>6.2f}   (true σ=15)")


**3 个异常值（0.3% 污染）把 std 从 15 拉到 21**，而 `1.4826×MAD` 稳如泰山。**生产环境的异常检测阈值应该用 MAD 不用 std**——否则异常值会"抬高自己藏身的门槛"。
0.3% contamination drags std from 15 to 21; MAD doesn't budge. Production outlier thresholds should use MAD — otherwise outliers raise the very bar meant to catch them.


<a id="3"></a>
## 3. ⭐ 为什么除 $n-1$ / Why Divide by $n-1$

**面试高频**。两层回答：

### 直觉层 / Intuition

$\bar{x}$ 本身就是从这份样本算的——**样本点天然离 $\bar{x}$ 比离真均值 $\mu$ 更近**：
$$\sum_i (x_i - \bar{x})^2 \;\le\; \sum_i (x_i - \mu)^2$$
所以用 $\bar{x}$ 算的偏差平方和**系统性偏小**，除 $n$ 会低估 $\sigma^2$。除 $n-1$ 恰好补回。

### 数学层 / The math

自由度：$n$ 个偏差 $(x_i - \bar{x})$ 受一条约束 $\sum_i (x_i - \bar{x}) = 0$，**只有 $n-1$ 个能自由变动**。

形式证明的核心一步：
$$\mathbb{E}\Big[\sum_i (x_i - \bar{x})^2\Big] = (n-1)\,\sigma^2$$

所以 $s^2 = \frac{1}{n-1}\sum (x_i-\bar{x})^2$ 满足 $\mathbb{E}[s^2] = \sigma^2$（无偏）。下面用模拟验证：


In [ ]:
# 模拟验证无偏性 / Simulate unbiasedness
true_var = 25.0          # σ = 5
n = 10                   # 小样本最能看出差别 / small n shows the gap best
n_experiments = 100_000

biased, unbiased = [], []
for _ in range(n_experiments):
    sample = rng.normal(0, 5, size=n)
    dev2 = np.sum((sample - sample.mean())**2)
    biased.append(dev2 / n)
    unbiased.append(dev2 / (n - 1))

print(f"true σ²              : {true_var}")
print(f"E[除以 n   的估计]    : {np.mean(biased):.3f}   ← 系统性低估 ~{(1-np.mean(biased)/true_var)*100:.0f}%")
print(f"E[除以 n-1 的估计]    : {np.mean(unbiased):.3f}   ← 无偏 ✓")
print(f"理论预测: 除 n 的期望 = (n-1)/n × σ² = {(n-1)/n*true_var:.3f}")


**模拟和理论完全吻合**：除 $n$ 平均低估 10%（$=1/n$），除 $n-1$ 正中靶心。

> ⚠ **numpy 的坑**：`np.std(x)` 默认 `ddof=0`（除 $n$）！统计推断永远写 `np.std(x, ddof=1)`。pandas 的 `.std()` 默认 `ddof=1`——**两个库默认值相反**，这是真实事故来源。
> numpy defaults to ddof=0, pandas to ddof=1 — opposite defaults, a real bug source.


<a id="4"></a>
## 4. 形状：偏度与峰度 / Skewness & Kurtosis

### 偏度 / Skewness —— 第三阶标准化矩

$$g_1 = \frac{\frac{1}{n}\sum (x_i - \bar{x})^3}{s^3}$$

- $g_1 > 0$：右偏（长尾在右，**均值 > 中位数**）——收入、房价、网页停留时长
- $g_1 < 0$：左偏——考试分数（天花板效应）
- $g_1 \approx 0$：对称

### 峰度 / Kurtosis —— 第四阶（**衡量尾部，不是峰的尖度！**）

$$g_2 = \frac{\frac{1}{n}\sum (x_i - \bar{x})^4}{s^4} - 3 \quad (\text{excess kurtosis, 正态} = 0)$$

- $g_2 > 0$：**重尾**（金融收益率典型 $g_2 \in [3, 30]$）→ 极端事件比正态预测的频繁得多
- $g_2 < 0$：轻尾（均匀分布 $g_2 = -1.2$）

> 💡 现代理解：**峰度 ≈ 尾部厚度**。"尖峰"是过时的错误直觉——峰度对中心几乎不敏感，对尾部极端值四次方放大。
> Modern reading: kurtosis ≈ tail weight. The "peakedness" story is outdated — the 4th power makes it a tail metric.


In [ ]:
# 三种形状对比 / Three shapes side-by-side
samples = {
    "Normal":          rng.normal(0, 1, 50_000),
    "LogNormal (右偏)": rng.lognormal(0, 0.6, 50_000),
    "Student-t df=3 (重尾)": rng.standard_t(3, 50_000),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, (name, s) in zip(axes, samples.items()):
    sk, ku = st.skew(s), st.kurtosis(s)        # scipy 默认就是 excess kurtosis
    ax.hist(s, bins=120, density=True, alpha=0.75)
    ax.set_title(f"{name}\nskew={sk:.2f}, excess kurt={ku:.2f}")
    ax.set_xlim(np.percentile(s, 0.2), np.percentile(s, 99.8))
plt.tight_layout(); plt.show()

# 偏态下 mean vs median 的错位 / Mean-median gap under skew
ln = samples["LogNormal (右偏)"]
print(f"LogNormal: mean={ln.mean():.3f} > median={np.median(ln):.3f}  ← 右偏的签名")


<a id="5"></a>
## 5. 标准化与 z 分数 / Standardization & z-scores

$$z_i = \frac{x_i - \bar{x}}{s}$$

把任何数据变成"均值 0、标准差 1"——**跨量纲比较**的通用货币：
- "身高 z=+2" 和 "考分 z=+2" 可以直接比（都是"高于均值 2 个标准差"）
- 经验法则（正态下）：$|z|>2$ 约 5%，$|z|>3$ 约 0.3% → 朴素异常值阈值
- ML 预处理的 `StandardScaler` 就是它（Part 3.4 详述）

> ⚠ 对**重尾/偏态**数据，z 分数的"2 倍标准差 = 罕见"直觉失效（第 4 节的金融数据 $|z|>3$ 家常便饭）——又回到"先看形状再选工具"。


<a id="6"></a>
## 6. 关联：三种相关系数 ⭐ / Three Correlations

| | Pearson $r$ | Spearman $\rho$ | Kendall $\tau$ |
|---|---|---|---|
| 衡量 | **线性**关系 | **单调**关系（对秩做 Pearson）| 单调（一致对比例）|
| 对异常值 | ❌ 敏感 | ✅ 稳健 | ✅ 最稳健 |
| 适用 | 双正态、线性 | 非线性单调、有序变量 | 小样本、多结点 |

$$r = \frac{\sum (x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum(x_i-\bar{x})^2}\sqrt{\sum(y_i-\bar{y})^2}}$$

**0.9 节的教训复习**：相关 = 0 ≠ 独立（$y=x^2$ 的反例）；这里再加一条：**相关 ≠ 因果**（冰淇淋销量与溺水人数）。


In [ ]:
# 三种相关在不同关系形态下的表现 / The three under different relationships
x = rng.uniform(0, 10, 300)

scenarios = {
    "线性 + 噪声":       2*x + rng.normal(0, 3, 300),
    "指数（单调非线性）":  np.exp(0.5*x) + rng.normal(0, 5, 300),
    "线性 + 5 个异常值":   np.where(np.arange(300) < 5, 100, 2*x + rng.normal(0, 3, 300)),
}

print(f"{'关系形态':<14} {'Pearson':>9} {'Spearman':>9} {'Kendall':>9}")
print("-" * 48)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, (name, y) in zip(axes, scenarios.items()):
    pr, sp, kd = st.pearsonr(x, y)[0], st.spearmanr(x, y)[0], st.kendalltau(x, y)[0]
    print(f"{name:<16} {pr:>8.3f} {sp:>9.3f} {kd:>9.3f}")
    ax.scatter(x, y, s=8, alpha=0.5); ax.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()


**读结果**：
- 指数关系：Pearson 被压低（不是直线），Spearman ≈ 1（完美单调）→ **怀疑非线性时用 Spearman**
- 5 个异常值（1.7% 污染）就把 Pearson 砸了一大块，Spearman 几乎不动

**选择口诀**：默认 Pearson；散点图弯了或有异常值 → Spearman；要给非技术人员讲"排序一致性" → Kendall。


<a id="7"></a>
## 7. ⚠ Anscombe 四重奏：只看数字会死 / Anscombe's Quartet

四组数据，**均值、方差、相关系数、回归线全部相同**（小数点后两位）——但形状天差地别。
Four datasets with identical means, variances, correlations, and regression lines — utterly different shapes.


In [ ]:
anscombe = sns.load_dataset("anscombe")

# 数字上四组几乎一模一样 / Numerically near-identical
print(anscombe.groupby("dataset").agg(
    x_mean=("x", "mean"), x_var=("x", "var"),
    y_mean=("y", "mean"), y_var=("y", "var"),
).round(2))
print("\ncorrelations:", {d: round(st.pearsonr(g.x, g.y)[0], 3)
                          for d, g in anscombe.groupby("dataset")})

g = sns.lmplot(data=anscombe, x="x", y="y", col="dataset", col_wrap=4,
               height=2.6, ci=None, scatter_kws={"s": 30})
g.fig.suptitle("Anscombe's Quartet — same stats, different stories", y=1.05)
plt.show()


**四个故事**：I 真线性；II 是抛物线（该用二次模型）；III 一条完美直线被一个异常值拐弯；IV 完全没关系，全靠一个杠杆点撑出相关。

**教训 = 本课程 EDA 哲学的根**：**描述统计和可视化必须成对出现**。0.5 节的图表技能不是装饰，是防身。
The lesson: summary statistics and plots must travel together. Visualization isn't decoration; it's defense.


<a id="8"></a>
## 8. 实战：Tips 完整描述报告 / Full Descriptive Report on Tips

把全课工具打包成一个**可复用的描述函数**，对 Tips 出一份报告。


In [ ]:
tips = sns.load_dataset("tips")
tips["tip_rate"] = tips["tip"] / tips["total_bill"]

def describe_plus(s: pd.Series) -> dict:
    # 比 .describe() 更全的描述 / richer than .describe()
    mad = np.median(np.abs(s - s.median()))
    return {
        "n": len(s), "mean": s.mean(), "median": s.median(),
        "trimmed10%": st.trim_mean(s, 0.1),
        "std": s.std(), "IQR": s.quantile(.75) - s.quantile(.25),
        "robust_σ(MAD)": 1.4826 * mad,
        "skew": st.skew(s), "ex_kurt": st.kurtosis(s),
        "P5": s.quantile(.05), "P95": s.quantile(.95),
    }

report = pd.DataFrame({c: describe_plus(tips[c])
                       for c in ["total_bill", "tip", "tip_rate"]}).T.round(3)
print(report)


In [ ]:
# 关联结构：数值列的两种相关对照 / Pearson vs Spearman on the numeric columns
num = tips[["total_bill", "tip", "size", "tip_rate"]]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, method in zip(axes, ["pearson", "spearman"]):
    sns.heatmap(num.corr(method=method), annot=True, fmt=".2f",
                cmap="coolwarm", vmin=-1, vmax=1, ax=ax, cbar=False)
    ax.set_title(f"{method} correlation")
plt.tight_layout(); plt.show()


**报告解读**（把数字翻译成话——DS 的核心技能）：
1. `total_bill` 右偏（skew≈1.1）：均值 19.8 > 中位数 17.8 → 报告"典型账单"应该说中位数
2. `tip_rate` 重尾（kurt≈3+）：存在极端慷慨的客人（最高 71%！0.5 节见过）→ 均值小费率会被高估
3. `total_bill ↔ tip` Pearson 0.68 ≈ Spearman 0.68 → 关系基本线性，没有异常值作怪
4. `tip_rate ↔ total_bill` 是**负**相关 → 账单越大小费率越低（大桌摊薄效应，与 0.5 节 EDA 一致）


<a id="9"></a>
## 9. 小结 / Summary

```
描述统计四象限
  位置: mean(脆) / median(稳) / trimmed mean(折中) — 看击穿点
  散布: std(脆) / IQR / 1.4826×MAD(最稳) — 异常检测用 MAD
  形状: skew(均值中位数错位方向) / kurtosis(尾部厚度，非尖度)
  关联: Pearson(线性) / Spearman(单调) / Kendall(秩一致)
```

### 💡 面试速查
1. **n-1**：自由度少一个（$\sum(x_i-\bar x)=0$ 的约束）；$\mathbb{E}[s^2]=\sigma^2$ 才无偏；模拟可证除 $n$ 低估 $1/n$
2. **均值 vs 中位数**：差得大 = 偏态信号；右偏 mean > median
3. **峰度是尾部**不是峰：金融数据 excess kurt 3-30
4. **numpy ddof=0 / pandas ddof=1** 默认相反
5. **Anscombe**：统计量相同形状不同 → 必须画图

### 下一节
**2.2 分布实战**——不再是"认识分布"（0.9 干过了），而是"**给真实数据挑分布、拟合、检验拟合好坏**"。
